In [1]:
import os
os.system("fuser -k 8000/tcp")
print("포트 8000 정리 완료")

포트 8000 정리 완료


In [ ]:
!pip install diffusers transformers accelerate safetensors fastapi uvicorn pyngrok python-multipart torch controlnet_aux

In [3]:
# ──────────────────────────────────────────────
# 셀 2: SDXL Base 단일 모델 (메모리 최적화 + 고품질)
# ──────────────────────────────────────────────
import os, io, base64, threading
import torch
from PIL import Image
from diffusers import StableDiffusionXLImg2ImgPipeline, DPMSolverMultistepScheduler
from flask import Flask, request, jsonify

os.system("fuser -k 8000/tcp 2>/dev/null; sleep 1")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("모델 로딩 중...")
pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True
)
pipe.enable_model_cpu_offload()   # VRAM 절약
pipe.enable_attention_slicing(1)  # 메모리 추가 절약
pipe.enable_vae_tiling()          # VAE OOM 방지
print("✅ 모델 로드 완료")

app = Flask(__name__)

NEGATIVE_PROMPT = (
    "multiple logos, grid, collage, frames, watermark, signature, text, letters, "
    "blurry, low quality, distorted, noisy, grainy, pixelated, "
    "multiple images, side by side, tiled, mosaic, panel, border, "
    "background pattern, shadow, gradient background, ugly, deformed"
)

@app.route("/generate", methods=["POST"])
def generate():
    try:
        prompt     = request.form.get("prompt", "a professional logo")
        num_images = int(request.form.get("num_images", 2))
        image_b64  = request.form.get("image_b64", None)

        if image_b64:
            img_bytes  = base64.b64decode(image_b64)
            init_image = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((1024, 1024))
        else:
            init_image = Image.new("RGB", (1024, 1024), (255, 255, 255))

        results = []
        for i in range(num_images):
            print(f"[{i+1}/{num_images}] 생성 중...")
            torch.cuda.empty_cache()  # 매 생성 전 캐시 비우기

            output = pipe(
                prompt=prompt,
                image=init_image,
                strength=0.75,
                guidance_scale=10.0,
                num_inference_steps=50,
                negative_prompt=NEGATIVE_PROMPT,
            ).images[0]

            buf = io.BytesIO()
            output.save(buf, format="PNG")
            results.append(base64.b64encode(buf.getvalue()).decode("utf-8"))
            print(f"[{i+1}/{num_images}] ✅ 완료")

        return jsonify({"images_b64": results})

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

# ControlNet 모델
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from controlnet_aux import HEDdetector

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble",
    torch_dtype=torch.float16
)
pipe_controlnet = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
)
pipe_controlnet.scheduler = UniPCMultistepScheduler.from_config(pipe_controlnet.scheduler.config)
pipe_controlnet.enable_model_cpu_offload()
pipe_controlnet.enable_attention_slicing(1)
print("✅ ControlNet 모델 로드 완료")

hed = HEDdetector.from_pretrained("lllyasviel/Annotators")

@app.route("/generate_file", methods=["POST"])
def generate_file():
    try:
        prompt = request.form.get("prompt", "a professional logo")
        num_images = int(request.form.get("num_images", 1))
        image_b64 = request.form.get("image_b64", None)

        if image_b64:
            img_bytes = base64.b64decode(image_b64)
            init_image = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((512, 512))
        else:
            init_image = Image.new("RGB", (512, 512), (255, 255, 255))

        control_image = hed(init_image, scribble=True)

        results = []
        for i in range(num_images):
            print(f"[{i+1}/{num_images}] 생성 중...")
            torch.cuda.empty_cache()

            output = pipe_controlnet(
                prompt=prompt,
                image=control_image,
                num_inference_steps=30,
                guidance_scale=9.0,
                negative_prompt=NEGATIVE_PROMPT,
            ).images[0]

            buf = io.BytesIO()
            output.save(buf, format="PNG")
            results.append(base64.b64encode(buf.getvalue()).decode("utf-8"))
            print(f"[{i+1}/{num_images}] ✅ 완료")

        return jsonify({"images_b64": results})

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

t = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8000))
t.daemon = True
t.start()
print("🚀 Flask 서버 실행 중 (port 8000)")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


모델 로딩 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2294: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionXLImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(


✅ 모델 로드 완료


ModuleNotFoundError: No module named 'controlnet_aux'

In [ ]:
# 셀 3: ngrok 터널
from pyngrok import ngrok
import time, requests

ngrok.set_auth_token("3EHkgXfCcPwIFKlUUZthANMYcjJ_7PWLNZyncrpiooiyuf9WS")  # ← 토큰은 여기에만

time.sleep(5)  # Flask 서버 시작 대기

public_url = ngrok.connect(8000).public_url
print(f"✅ ngrok URL: {public_url}")
print(f"👉 .env의 COLAB_SD_URL 에 이 값을 붙여넣으세요:\n   {public_url}")

# 헬스체크
try:
    r = requests.get(f"{public_url}/health", timeout=10)
    print(f"서버 상태: {r.json()}")
except Exception as e:
    print(f"헬스체크 실패 (서버 아직 시작 중일 수 있음): {e}")

✅ ngrok URL: https://unspoken-gloss-pants.ngrok-free.dev
👉 .env의 COLAB_SD_URL 에 이 값을 붙여넣으세요:
   https://unspoken-gloss-pants.ngrok-free.dev
헬스체크 실패 (서버 아직 시작 중일 수 있음): Expecting value: line 1 column 1 (char 0)


In [ ]:
# 셀 4: Flask 로컬 헬스체크
import requests

try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print("✅ Flask 응답:", r.json())
except Exception as e:
    print("❌ Flask 응답 없음:", e)

INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 07:50:21] "GET /health HTTP/1.1" 200 -


✅ Flask 응답: {'status': 'ok'}


In [ ]:
import requests
try:
    r = requests.post("http://localhost:8000/generate",
                      data={"prompt": "test logo", "num_images": 1},
                      timeout=120)
    print("상태:", r.status_code)
    print("응답:", r.text[:1000])
except Exception as e:
    print("에러:", e)

에러: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7dad8d809fd0>: Failed to establish a new connection: [Errno 111] Connection refused'))
